In [ ]:
# SVC_HyperOpt
from hyperopt import fmin, tpe, STATUS_OK, Trials
from sklearn.model_selection import cross_val_score
import numpy as np

# Search Space 정의
lsvc_search_space = {
    'C': hp.loguniform('C', np.log(0.001), np.log(1000)),  # 정규화 강도 (작을수록 강한 정규화)
    'class_weight': hp.choice('class_weight', [None, 'balanced']),  # 클래스 가중치
    'max_iter': hp.quniform('max_iter', 1000, 10000, 1000),  # 최대 반복 횟수
    'tol': hp.loguniform('tol', np.log(1e-5), np.log(1e-2)),  # 수렴 허용 오차
    'dual': hp.choice('dual', [False, True]),  # dual formulation (n_samples > n_features일 때 False 권장)
}

# Objective 함수
def objective(params):
    # 파라미터 타입 변환
    params['C'] = float(params['C'])
    params['max_iter'] = int(params['max_iter'])
    params['tol'] = float(params['tol'])
    
    # 모델 생성
    model = LinearSVC(
        C=params['C'],
        class_weight=params['class_weight'],
        max_iter=params['max_iter'],
        tol=params['tol'],
        dual=params['dual'],
        random_state=team_rs
    )
    
    # 교차 검증
    try:
        scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')  # 또는 'f1'
        score = scores.mean()
    except Exception as e:
        print(f"Error: {e}")
        return {'loss': 1.0, 'status': STATUS_OK}
    
    # HyperOpt는 최소화하므로 음수로 반환
    return {'loss': -score, 'status': STATUS_OK}

# 최적화 실행
trials = Trials()
best_params = fmin(
    fn        = objective,
    space     = lsvc_search_space,
    algo      = tpe.suggest,
    max_evals = 50,  # 시도 횟수
    trials    = trials,
    rstate    = np.random.default_rng(42)
)

print("Best parameters:", best_params)

# 최적 파라미터로 최종 모델 학습
# best_params에서 파라미터 추출 및 변환
final_params = {
    'C': float(best_params['C']),
    'class_weight': [None, 'balanced'][best_params['class_weight']],  # choice는 인덱스로 반환됨
    'max_iter': int(best_params['max_iter']),
    'tol': float(best_params['tol']),
    'dual': [False, True][best_params['dual']],  # choice는 인덱스로 반환됨
}

# 최종 모델 학습
final_model = LinearSVC(**final_params, random_state=42)
final_model.fit(X_train, y_train)

# 예측
y_pred = final_model.predict(X_test)

# 평가
from sklearn.metrics import classification_report, roc_auc_score
print(classification_report(y_test, y_pred))